In [1]:
import dspy
import os
from dotenv import load_dotenv
from openinference.instrumentation.dspy import DSPyInstrumentor
from langfuse import get_client

load_dotenv("/home/azureuser/cloudfiles/code/email-generation/.env.local", override=True)

os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_BASE_URL"] = "http://localhost:3000"

langfuse = get_client()
import openlit
openlit.init(tracer=langfuse._otel_tracer, disable_batch=True)

In [ ]:
api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_base = os.getenv("AZURE_LANGUAGE_ENDPOINT")
api_version = os.getenv("AZURE_API_VERSION")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

lm_4o_mini = dspy.LM(
    model=f"azure/{os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')}",
    api_key=api_key,
    api_base=api_base,
    # api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0.8, 
    model_type='responses'
)

lm_5_mini = dspy.LM( 
    model=f"azure/gpt-5-mini",
    api_key=os.getenv("AZURE_OPENAI_4_1_KEY"),
    api_base=api_base,
    # api_version="2025-04-14",
    temperature=1, 
    model_type='responses'
)

lm_41_mini = dspy.LM(
    model=f"azure/gpt-4.1-mini",
    api_key=os.getenv("AZURE_OPENAI_4_1_KEY"),
    api_base=api_base,
    # api_version="2025-04-14",
    temperature=0.4, 
    model_type='responses'
)

# mistral = dspy.LM(
#     model="openai/mistral-small-2503",
#     api_key=os.getenv("AZURE_MISTRAL_KEY"),
#     api_base="https://anuj-ai-resource.services.ai.azure.com/openai/v1/",
#     api_version="2024-05-01-preview",
#     temperature=0.8, 
#     # model_type='responses'
# )

# phi4 = dspy.LM(
#     model="openai/Phi-4-reasoning",
#     api_key=os.getenv("AZURE_MISTRAL_KEY"),
#     api_base="https://anuj-ai-resource.services.ai.azure.com/openai/v1/",
#     api_version="2024-05-01-preview",
#     temperature=0.8, 
#     # model_type='responses'
# )

dspy.configure(lm=lm_41_mini)

In [3]:
import pandas as pd

# matter_level_df = pd.read_csv("/home/azureuser/cloudfiles/code/production-data/8-Dec-2025/processed-data/matter-level-df.csv", 
#                               parse_dates=['submitted_on', 'due_on', 'created_on', 'activity_created_on'])
# matter_info = pd.read_csv("/home/azureuser/cloudfiles/code/production-data/8-Dec-2025/processed-data/matter-info.csv",
#                           parse_dates=['last_invoiced_on', 'last_paid_on' ,'last_statement_sent_on'])

#date columns: 
# matter_level_df -  submitted_on, due_on, created_on, activity_created_on, 
# matter_info - last_invoiced_o, last_paid_on ,last_statement_sent_on,  

PREFIX = '/home/azureuser/cloudfiles/code/production-data/12-dec-25/tenant1-12-20-25'
OUTPUT_FILE = f'{PREFIX}-MASTER-REPORT.csv'

# invoices.csv: 100 samples
# #['client_id', 'matter_id', 'invoice_number', 'invoice_amount', 'ar_amount', 'submitted_on', 'due_on', 'days_overdue', 'status']
df_invoices = pd.read_csv(f'{PREFIX}-invoices.csv') 

# activities.csv: 
# #['client_id', 'matter_id', 'invoice_number', 'created_on', 'event_type', 'name', 'description']
df_activities = pd.read_csv(f'{PREFIX}-activities-REDACTED.csv', parse_dates = ['created_on'], date_format='%Y-%m-%d %H:%M:%S.%f%z')

# notes.csv: 
# #['client_id', 'matter_id', 'invoice_number', 'created_on', 'content']
df_notes = pd.read_csv(f'{PREFIX}-notes-REDACTED.csv', parse_dates=['created_on'])
df_notes['created_on'] = pd.to_datetime(df_notes['created_on'], format = 'mixed')
df_notes = df_notes.dropna()
df_notes['invoice_number'] = df_notes['invoice_number'].apply(lambda x: int(x))
df_notes.head()
# df_notes['invoice_number'] = df_notes['invoice_number'].astype(str)

# matters.csv: 
# #['client_id', 'matter_id', 'wip_amount', 'last_invoiced_on', 'last_paid_on', 'last_statement_sent_on', 'payment_terms_in_days', 'total_billed_amount', 'total_collected_amount', 'total_work_amount', 'ar_total', 'ar_overdue']
df_matters = pd.read_csv(f'{PREFIX}-matters.csv', parse_dates=['last_invoiced_on', 'last_statement_sent_on'], date_format='%Y-%m-%d %H:%M:%S.%f%z')

# clients.csv: 
# #['client_id', 'wip_amount', 'last_invoiced_on', 'last_action_on', 'last_paid_on', 'last_statement_sent_on', 'payment_terms_in_days', 'total_billed_amount', 'total_collected_amount', 'total_work_amount', 'ar_total', 'ar_overdue']
df_clients = pd.read_csv(f'{PREFIX}-clients.csv', parse_dates=['last_invoiced_on', 'last_action_on', 'last_paid_on', 'last_statement_sent_on'], date_format='%Y-%m-%d %H:%M:%S.%f%z')

df_messages = pd.read_csv(f'{PREFIX}-messages-REDACTED.csv')

In [4]:
df_messages

,client_id,sent_on,subject,body
0,16,2025-09-26 14:35:04+00,RE: 2nd REQUEST: Legal invoice from <Organizat...,"Thank you again for all of your assistance, Jo..."
1,562116,2025-10-29 15:31:20+00,RE: Your October 2025 Invoice from <Organization>,"Invoice Submission (External)Hi <Person>, I ho..."
2,567518,2025-08-28 19:35:07+00,Re: **REPLY REQUESTED** <Organization> (056751...,[EXTERNAL EMAIL]\ndone!\napologies for the del...
3,567518,2025-08-28 12:34:41.720362+00,RE: **REPLY REQUESTED** <Organization>/<Organi...,Good morning <Person> Could I please ask you t...
4,580679,2025-12-02 19:08:27+00,Re: Husch Blackwell Invoice - <Organization>,[EXTERNAL EMAIL]\nSuper - thanks! <Person>. +4...


In [5]:
df_notes

,client_id,matter_id,invoice_number,created_on,content
1953,16,0000016-0000163,3858831,2025-11-25 19:29:53.511774+00:00,Waiting for AR to add the unallocated on the b...
1954,16,0000016-0000163,3814158,2025-09-18 20:23:50.024062+00:00,"<DateTime> email to <Person> for claim number,..."
1955,16,0000016-0000164,3814159,2025-09-18 20:24:16.115800+00:00,"<DateTime> email to <Person> for claim number,..."
2150,2899,0002899-0000034,3741724,2025-09-16 21:17:31.990738+00:00,<DateTime> - K<Person> - client has requested ...
2151,2899,0002899-0000035,3741755,2025-09-16 21:21:47.080575+00:00,<DateTime> - <Person> - Client response states...
...,...,...,...,...,...
7461,6053054,6053054-0000099,3830477,2025-10-24 14:26:35.535868+00:00,<DateTime> updated cost narratives to include ...
7462,6053054,6053054-0000099,3808607,2025-09-19 20:41:12.948517+00:00,"<DateTime> per <Person>, <Person> to email to ..."
7463,6053054,6053054-0000100,3868113,2025-12-15 18:03:29.562652+00:00,<DateTime> manually sent updated copy of invoi...
7519,6053056,6053056-0000001,3786289,2025-08-07 15:57:07.469024+00:00,Invoice delivery pending update of invoice del...


In [6]:
df_summaries = pd.read_csv("/home/azureuser/cloudfiles/code/production-data/client_summarization_stored_completions_5th_feb26.csv")
df_summaries.head()

,completion_id,input_data,output_data
0,chatcmpl-D5OhEPBr9feteTKHFVqVPDGomJmFo,SyncCursorPage[ChatCompletionStoreMessage](dat...,"{\n ""id"": ""chatcmpl-D5OhEPBr9feteTKHFVqVPDGom..."
1,chatcmpl-D5Ew1ShgFECUweV0bORKx04cWf8uA,SyncCursorPage[ChatCompletionStoreMessage](dat...,"{\n ""id"": ""chatcmpl-D5Ew1ShgFECUweV0bORKx04cW..."
2,chatcmpl-D5EObN6Z1kzjxrZU2Mx7Wd0OkYq0p,SyncCursorPage[ChatCompletionStoreMessage](dat...,"{\n ""id"": ""chatcmpl-D5EObN6Z1kzjxrZU2Mx7Wd0Ok..."
3,chatcmpl-D5ENMYYiOXfT2iEM23SLnJ21at8M1,SyncCursorPage[ChatCompletionStoreMessage](dat...,"{\n ""id"": ""chatcmpl-D5ENMYYiOXfT2iEM23SLnJ21a..."
4,chatcmpl-D5BZKOuyVuimv1c0g4dKPJzdmTNZR,SyncCursorPage[ChatCompletionStoreMessage](dat...,"{\n ""id"": ""chatcmpl-D5BZKOuyVuimv1c0g4dKPJzdm..."


In [7]:
df_summaries.shape

(20, 3)

In [8]:
import ast
import json
import re

MSG_RE = re.compile(r"ChatCompletionStoreMessage\(content='((?:\\.|[^'])*)'.*?role='(.*?)'", re.DOTALL)

def _unescape_python_string(raw: str) -> str:
    try:
        return ast.literal_eval("'" + raw + "'")
    except Exception:
        return bytes(raw, 'utf-8').decode('unicode_escape', errors='replace')

def _split_user_content(content: str):
    if not content:
        return '', '', ''
    normalized = content.replace('\r\n', '\n')
    fin_key = 'Financial data:\n'
    notes_key = '\n\nNotes:\n'
    if fin_key not in normalized:
        return normalized.strip(), '', ''
    prompt, rest = normalized.split(fin_key, 1)
    financial_data = rest
    notes = ''
    if notes_key in rest:
        financial_data, notes = rest.split(notes_key, 1)
    return prompt.strip(), financial_data.strip(), notes.strip()

def parse_input_data(input_data: str):
    if not isinstance(input_data, str):
        return {'system_prompt': '', 'user_prompt': '', 'financial_data': '', 'notes': ''}
    messages = []
    for match in MSG_RE.finditer(input_data):
        raw = match.group(1)
        role = match.group(2)
        content = _unescape_python_string(raw)
        messages.append((role, content))
    system_prompt = '\n\n'.join([c for r, c in messages if r == 'system']).strip()
    user_content = '\n\n'.join([c for r, c in messages if r == 'user']).strip()
    user_prompt, financial_data, notes = _split_user_content(user_content)
    return {
        'system_prompt': system_prompt,
        'user_prompt': user_prompt,
        'financial_data': financial_data,
        'notes': notes,
    }

def parse_output_data(output_data: str) -> str:
    if not isinstance(output_data, str) or not output_data:
        return ''
    cleaned = re.sub(r':\s*<([^>]+)>', r': "<\1>"', output_data)
    try:
        payload = json.loads(cleaned)
        return payload.get('choices', [{}])[0].get('message', {}).get('content', '')
    except Exception:
        match = re.search(r'\"content\"\s*:\s*\"((?:\\.|[^\"])*)\"', output_data, re.DOTALL)
        if not match:
            return ''
        return bytes(match.group(1), 'utf-8').decode('unicode_escape', errors='replace')

parsed_rows = df_summaries['input_data'].apply(parse_input_data)
df_parsed = pd.DataFrame(list(parsed_rows))
df_extracted = pd.concat([
    df_summaries[['completion_id']].copy(),
    df_parsed,
    df_summaries['output_data'].apply(parse_output_data).rename('output_summary'),
], axis=1)


df_extracted.head()


,completion_id,system_prompt,user_prompt,financial_data,notes,output_summary
0,chatcmpl-D5OhEPBr9feteTKHFVqVPDGomJmFo,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 1 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,CreatedOn","The client currently has **$20,806.67** in ove..."
1,chatcmpl-D5Ew1ShgFECUweV0bORKx04cWf8uA,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 3 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,Create...","The client currently has **$16,373.00** in ove..."
2,chatcmpl-D5EObN6Z1kzjxrZU2Mx7Wd0OkYq0p,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 2 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,Create...","The client currently has **$6,096.75** in over..."
3,chatcmpl-D5ENMYYiOXfT2iEM23SLnJ21at8M1,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 2 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,Create...","The client currently has **$6,096.75** in over..."
4,chatcmpl-D5BZKOuyVuimv1c0g4dKPJzdmTNZR,"You are Oddr Summarizer, an AI assistant that ...",Given a client with the following financial da...,"The client has 3 active invoices, lifetime bil...","ClientId,MatterId,InvoiceNumber,Content,Create...","The client currently has **$16,373.00** in ove..."


In [9]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")
df_extracted['notes_token_count'] = df_extracted['notes'].apply(lambda x: len(encoding.encode(x)))

In [10]:
# df_extracted.to_csv("/home/azureuser/cloudfiles/code/production-data/stored_completions_5-feb.csv", index = False)

In [11]:
notes = df_extracted.notes.tolist()

In [12]:
df_extracted.financial_data.tolist()[2]

'The client has 2 active invoices, lifetime billed amount of $10,037.91, lifetime collected amount of $3,941.16, WIP of $0.00, overdue AR of $6,096.75, outstanding AR of $6,096.75, average DSO of 260.'

In [13]:
df_extracted.notes.tolist()[1]

'ClientId,MatterId,InvoiceNumber,Content,CreatedOn\n033546,,,"Forwarded 1<DateTime> email to client re invoices <PhoneNumber>, <PhoneNumber> &amp; <PhoneNumber> and if we need to make further changes, in light of them paying the December and January invoices.",02/03/2026\n033546,,,"Forwarded 1/6 email to K. Borowik asking him to fix the address and I will send to client. Sent revised invoices to client, asking if that works for them.",<DateTime>\n033546,,,"Emailed K. Borowik asking him to revise invoices <PhoneNumber>, <PhoneNumber> &amp; 585916 and I will send to client asking if that format is acceptable. (Putting the property address in the client address.)",01/06/2026\n033546,,,"Emailed L. Schleicher re September invoices <PhoneNumber>, <PhoneNumber> L<Person> resubmitted <DateTime>, noting they paid a November invoice <DateTime>. Emailed client re same.",<DateTime>\n033546,,,Emailed L. Schleicher re client paid the October invoices leaving two September invoices open. <Person> rep

In [14]:
df_extracted.output_summary.tolist()[0]

'The client currently has **$20,806.67** in overdue accounts receivable with no payments collected or billed to date, indicating a significant delay in payment processing. There are no recent interactions or updates noted, suggesting a lack of follow-up on this outstanding invoice. Given the average Days Sales Outstanding (DSO) of **4**, immediate outreach is recommended to address the overdue balance and initiate payment discussions. It is crucial to monitor this account closely to prevent further delays and ensure timely resolution.'

In [15]:
import numpy as np

overlapping_invoices = np.intersect1d(df_notes.invoice_number, df_invoices.invoice_number.tolist()).tolist()
print("Total: ", len(overlapping_invoices), overlapping_invoices)

Total:  43 ['3750401', '3764007', '3769975', '3778983', '3779715', '3781783', '3782861', '3787068', '3790207', '3791993', '3792487', '3794224', '3794436', '3794925', '3795448', '3800269', '3800572', '3802099', '3808989', '3811927', '3812559', '3814420', '3815886', '3816783', '3817152', '3819232', '3819674', '3821183', '3825917', '3828055', '3834273', '3835452', '3839511', '3841423', '3848860', '3851985', '3853210', '3854555', '3858890', '3860477', '3864673', '3868951', '3873403']


In [16]:
import dspy
from typing import Literal, List

class ClientSummarization(dspy.Signature):
    """
    You are Oddr Summarizer, an AI assistant that helps law firms manage their Accounts Receivable. Your job is to generate clear, concise, and helpful summaries for attorneys using only the facts provided.

    You may infer logical next steps or patterns (e.g., payment gaps, lack of follow-up), but do not fabricate or invent any client interactions or internal updates that aren’t explicitly present in the input.

    If no notes are available, avoid referring to interactions. Focus only on AR status and suggest possible outreach or monitoring based on the financial data.

    Your tone should be professional, factual, and efficient — like a trusted human AR professional writing a briefing. Favor direct, punchy sentences over formal or padded language.

    Given a client with the following financial data, notes and email conversations across the client, its matters, and invoices, please summarize the client in a way that will make it digestible and meaningful to the time-constrained Attorney responsible for the client so they can understand what is going on with their AR and who is responsible for any next steps.

    Frame this as a timeline, starting with an overview of the client’s current AR status, then highlighting the most recent interaction and any key themes from earlier interactions — but only if such notes are present. Do not fabricate or infer specific updates. However, it’s acceptable to suggest likely follow-up steps or highlight the absence of recent activity if that pattern is evident.

    Reduce repetitive details by grouping similar updates, eliminate specific payment amounts where appropriate, and keep the summary focused and efficient. Mention past events only if they are essential to understanding the current delay or next step. Avoid repeating older context that does not affect present action.

    Format the summary as a single paragraph without any headings or titles, and use smart **bolding** to highlight key points. 
    Limit the summary to **4 complete, efficient sentences**. Favor punchy, high-signal writing over formal explanations. 
    **Do not use headings, bullet points, or labels.** **Mention date always like Jan 01,2025**
    
    Structure the summary as a single flowing paragraph that follows this order:
    [AR overview]
    [Highlights from recent interactions — only if present]
    [Relevant context from earlier interactions — only if helpful]
    [Next steps or follow-up actions]
    """

    financial_data: str = dspy.InputField(
        desc="Structured csv containing Invoice Numbers, Dates, Amounts Outstanding, "
             "and total balance. These figures are immutable facts."
    )

    notes: str = dspy.InputField(
        desc="Mixed timeline of client interaction, internal attorney notes, and billing "
             "status updates. Use for context reasoning, personalization"
    ) 

    email_conversation = dspy.InputField(
        desc="Email exchanges between the client and the attorney/biller/collector. Use for context reasoning and personalization."
    )

    summary: str = dspy.OutputField(
        desc="Actionable summary of the Client across the invoices, notes and financial_data. Accurately mention dates, invoice numbers, amounts when summarizing notes and financial_data.  It should be helpful for the attorneys and provides a up-to-date detailed overview of the the account. "
    )


summarizer = dspy.Predict(ClientSummarization)
summarizer_cot = dspy.ChainOfThought(ClientSummarization)


In [17]:
class InternalContextCompression(dspy.Signature):
    """
    You are Oddr Summarizer, an AI assistant that helps law firms manage their Accounts Receivable. Your job is to generate clear, concise, and helpful summaries for attorneys using only the facts provided.

    You may infer logical next steps or patterns (e.g., payment gaps, lack of follow-up), but do not fabricate or invent any client interactions or internal updates that are not explicitly present in the input.

    If no notes are available, avoid referring to interactions. Focus only on AR status and suggest possible outreach or monitoring based on the financial data.

    Your tone should be professional, factual, and efficient, like a trusted human AR professional writing a briefing. Favor direct, punchy sentences over formal or padded language.

    Compress the provided internal context for downstream client summarization. Frame it as a timeline of events and Keep critical dates, invoice numbers, status changes, promises, blockers, owner/action details, and maintain timeline continuity. Group repetitive events and remove filler.

    Return a single paragraph, no headings, no bullets, and cap output at #target_words words.
    """

    internal_context: str = dspy.InputField(desc="Large mixed internal context from notes and activities.")
    target_words: int = dspy.InputField(desc="Maximum number of words allowed in compressed output.")
    compressed_notes: str = dspy.OutputField(desc="Compressed context string used as notes input for ClientSummarization.")


internal_context_compressor = dspy.Predict(InternalContextCompression)

In [18]:
def data_formatter(df_invoices, df_notes, df_activities, invoice_number):

    notes = df_notes.query(f"invoice_number == {invoice_number}")[['created_on', 'content']].to_records(index=False)
    invoice_data = df_invoices.query(f"invoice_number=={invoice_number}").to_json(index = False, lines = True, orient = 'records')
    activities = df_activities.query(f"invoice_number=={invoice_number}").dropna().drop(columns = ['client_id', 'matter_id', 'invoice_number']).to_markdown(index = False)
    
    formatted_text = f"""
    ### INVOICE DATA: 
    {invoice_data}

    ### NOTES: 
    {notes}

    ### ACTIVITIES: 
    {activities}
    """
    return formatted_text, notes, invoice_data, activities

def data_formatter_csv(df_invoices, df_notes, df_activities, invoice_number):

    notes = df_notes.query(f"invoice_number == {invoice_number}")[['created_on', 'content']].to_csv(index=False)
    invoice_data = df_invoices.query(f"invoice_number=='{invoice_number}'").to_csv(index = False)
    activities = df_activities.query(f"invoice_number=='{invoice_number}'").dropna().drop(columns = ['client_id', 'matter_id', 'invoice_number']).to_csv(index = False)
    
    formatted_text = f"""
    ## INVOICE_NUMBER: {invoice_number}
    ### INVOICE DATA: 
    {invoice_data}

    ### NOTES: 
    {notes}

    ### ACTIVITIES: 
    {activities}

    //
    """
    # return formatted_text
    return formatted_text, notes, invoice_data, activities


def data_formatter_csv_list(df_invoices, df_notes, df_activities, invoice_numbers):

    if not isinstance(invoice_numbers, list):
        invoice_numbers = [invoice_numbers]
    
    invoice_data = df_invoices.query("invoice_number in @invoice_numbers").drop(columns=['client_id', 'matter_id']).to_csv(index=False)
    
    activities = df_activities.query("invoice_number in @invoice_numbers") \
                              .dropna() \
                              .drop(columns=['client_id', "matter_id"]) \
                              .to_csv(index=False)
    
    invoice_numbers = [int(inv_num.split("-")[0]) for inv_num in invoice_numbers]
    # print(invoice_numbers)
    notes = df_notes.query("invoice_number in @invoice_numbers")[[ 'invoice_number', 'created_on', 'content']].to_csv(index=False)
    formatted_text = f"""
    ### INVOICE DATA: 
    {invoice_data}

    ### NOTES: 
    {notes}

    ### ACTIVITIES: 
    {activities}
    """
    return formatted_text, notes, invoice_data, activities




In [19]:
def split_internal_context(internal_context, max_chars=7000, overlap_chars=350):
    text = (internal_context or "").strip()
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0
    step = max_chars - overlap_chars
    if step <= 0:
        step = max_chars

    while start < len(text):
        end = min(start + max_chars, len(text))
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start += step
    return chunks


def compress_internal_context(internal_context, target_words=200, chunk_chars=7000, overlap_chars=350):
    if not internal_context or not internal_context.strip():
        return ""

    chunks = split_internal_context(internal_context, max_chars=chunk_chars, overlap_chars=overlap_chars)
    if not chunks:
        return ""

    per_chunk_words = max(60, min(140, max(1, target_words // max(1, len(chunks)))))
    chunk_summaries = []

    for idx, chunk in enumerate(chunks, start=1):
        chunk_input = f"Chunk {idx}/{len(chunks)}:\n{chunk}"
        response = internal_context_compressor(
            internal_context=chunk_input,
            target_words=per_chunk_words,
        )
        summary = response.compressed_notes.strip()
        if summary:
            chunk_summaries.append(summary)

    if not chunk_summaries:
        return ""

    merged_context = "\n".join(chunk_summaries)
    final_response = internal_context_compressor(
        internal_context=merged_context,
        target_words=target_words,
    )
    compressed = final_response.compressed_notes.strip()

    if len(compressed.split()) > target_words:
        compressed = internal_context_compressor(
            internal_context=compressed,
            target_words=target_words,
        ).compressed_notes.strip()

    if len(compressed.split()) > target_words:
        compressed = " ".join(compressed.split()[:target_words]).strip()

    return compressed

In [21]:
df_notes.query("client_id == 562116").invoice_number.tolist()

[3825917, 3825917, 3851985]

In [23]:
i = 1

invoice_numbers = df_invoices.groupby(by = "matter_id").agg(list).invoice_number.tolist()
all_invoice_numbers_list = [lst for lst in invoice_numbers if len(lst)>=2]
invoice_numbers = all_invoice_numbers_list[i]
# invoice_numbers = ['3814159', '3814158', '3858831']
invoice_numbers = ['3825917', '3825917', '3851985']
print(invoice_numbers)
# print("Invoice numbers: ", invoice_numbers)
# # formatted_data, notes, invoice_data, activities = data_export_format(df_invoices, df_notes, df_activities, invoice_number)
# # formatted_data, notes, invoice_data, activities = data_formatter_csv(df_invoices, df_notes, df_activities, invoice_number)
# # formatted_data = matter_data_formatter_csv(df_invoices, df_notes, df_activities, invoice_numbers)
formatted_data, notes, invoice_data, activities = data_formatter_csv_list(df_invoices, df_notes, df_activities, invoice_numbers)
print(formatted_data)
print(df_messages.query("client_id == 562116").body.tolist())
# notes, invoice_data = df_extracted['notes'].tolist()[i], df_extracted['financial_data'].tolist()[i]
# internal_context = f"""### Invoice_data:\n{invoice_data} \n\n### NOTES:\n{notes}"""
# print(internal_context)

['3825917', '3825917', '3851985']

    ### INVOICE DATA: 
    invoice_number,invoice_amount,ar_amount,submitted_on,due_on,days_overdue,status
3825917,15137.5,15137.5,2025-10-16 20:55:36.604636+00,2025-11-08,42,Viewed
3851985,8080.0,8080.0,2025-11-18 13:54:44.687369+00,2025-12-17,3,Viewed


    ### NOTES: 
    invoice_number,created_on,content
3825917,2025-10-16 14:36:21.660739+00:00,I sent <Person> a follow up email that I need him to electronically sign each invoice page before I can send the invoice to the client.
3825917,2025-10-09 20:28:22.562291+00:00,I emailed <Person> to sign off on the invoice before it can go out to the client.
3851985,2025-11-17 16:15:11.826512+00:00,I emailed <Person> to add his signature to the invoice before emailing it to the client.


    ### ACTIVITIES: 
    invoice_number,created_on,event_type,name,description
3825917,2025-10-28 08:02:33+00,Event,Email Opened,Opened by <Email>
3825917,2025-10-28 00:58:19+00,Event,Email Opened,Opened by <Email>
3825917,

In [24]:
email_conversation = df_messages.query("client_id == 562116").body.tolist()
email_conversation

['Invoice Submission (External)Hi <Person>, I hope this email finds you well! I have attached invoices 3782330 and 3794119, which were <DateTime> billed in July and August. If you have any questions or need further assistance, please don\'t hesitate to reach out. Thank you, and have a great day! <Person>\u200b\u200b\u200b\u200bBilling SpecialistDirect: <Email> From: <Person> 彭帅 <<Email>> Sent: <DateTime> 2025 1<DateTime> AMTo: <Person>, <Person> <<Email>>; <Person> <<Email>>Cc: <Person> <<Email>>; <Person> <Person> <<Email>>Subject: FW: Your October 2025 Invoice from Husch Blackwell [EXTERNAL EMAIL] <Person> and <Person>: Could you please help with it? Thanks Best regards,<Person>发件人: <Person> <Person> <<Email>> 发送时间: <DateTime>收件人: <Email>抄送: <Person> <<Email>>; <Person>_<Person> <Person> <<Email>>主题: 转发: Your October 2025 Invoice from Husch Blackwell Hi Conley, I received the attached invoice <DateTime> and was surprise to find an outstanding balance on the bill. As I haven\'t receiv

In [27]:
with dspy.context(lm = lm_41_mini.copy(cache = False, temperature = 0.3)):
    response = summarizer_cot(financial_data = invoice_data, notes = notes, email_conversation = email_conversation)
    print(response)

Prediction(
    reasoning='The client currently has two outstanding invoices: invoice 3825917 for $15,137.50, overdue by 42 days as of Nov 08, 2025, and invoice 3851985 for $8,080.00, due Dec 17, 2025, and only 3 days overdue. Both invoices have been viewed by the client. The notes indicate repeated attempts to obtain electronic signatures from a contact person before the invoices could be sent to the client, with the last signature request for invoice 3851985 on Nov 17, 2025. The email conversation reveals the client was surprised by the outstanding balance on the October 2025 invoice and requested copies of the invoices, indicating some confusion or lack of receipt of the invoices. There is no evidence of payment or further client acknowledgment beyond viewing and signature requests. Next steps likely involve confirming receipt of the signed invoices and following up with the client to resolve the overdue balance on invoice 3825917 and ensure timely payment of invoice 3851985.',
    

In [ ]:
compressed_notes = compress_internal_context(internal_context, target_words=200)
notes_for_summary = compressed_notes if compressed_notes else notes
with dspy.context(lm = lm_41_mini.copy(cache = False, temperature = 0.4)):
    response = summarizer_cot(financial_data = invoice_data, notes = notes_for_summary)

In [45]:
print(compressed_notes)

Client has $16,373 overdue AR across 3 active invoices with an average DSO of 62. September and October invoices remain unpaid despite payments on November and January. Repeated emails from late 2024 to early 2026 addressed payment status and invoice corrections due to address issues. Contact updates occurred with new owners assigned. Client cited multi-level approval delays but promised payment; follow-ups planned if December invoice remains unpaid by 3/15/2026. Separately, a $16,816.29 balance from August to December 2023 is unpaid with no payments received. Multiple emails sent to internal contacts from October to December 2023; client described as "really scattered," but internal contacts committed to pressing client after receiving documents. Client agreed to check payment status; follow-up outreach ongoing with no confirmed payments yet.


In [46]:
response

Prediction(
    reasoning='The client currently has $16,373 in overdue accounts receivable across three active invoices, contributing to an average DSO of 62 days. Despite payments in November and January, the September and October invoices remain unpaid, indicating partial but inconsistent payment behavior. Multiple communications from late 2024 through early 2026 have focused on payment status, invoice corrections, and client contact updates, with the client citing multi-level approval delays but promising payment. There is also a separate older balance of $16,816.29 from August to December 2023 that remains unpaid, with ongoing follow-ups but no confirmed payments. The client is described as "really scattered," and internal contacts have committed to pressing the client further. Follow-up is planned if the December invoice remains unpaid by March 15, 2026, indicating a need for continued monitoring and outreach.',
    summary='The client has **$16,373 overdue across three active inv

In [ ]:
from tqdm import tqdm 
import time

invoice_numbers = df_invoices.groupby(by = "matter_id").agg(list).invoice_number.tolist()
all_invoice_numbers_list = [lst for lst in invoice_numbers if len(lst)>=2]

for i in tqdm(range(len(all_invoice_numbers_list))):
    invoice_numbers = all_invoice_numbers_list[i]
    formatted_data, notes, invoice_data, activities = data_formatter_csv_list(df_invoices, df_notes, df_activities, invoice_numbers)
    internal_context = f"""### NOTES:\n{notes}\n\n### ACTIVITIES:\n{activities}"""
    compressed_notes = compress_internal_context(internal_context, target_words=200)
    notes_for_summary = compressed_notes if compressed_notes else notes
    with dspy.context(lm = lm_41_mini.copy(cache = False, temperature = 0.4)):
        response = summarizer_cot(financial_data = invoice_data, notes = notes_for_summary)
        # print(response)
    time.sleep(5)

  0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           | 0/46 [00:00<?, ?it/s]

 11%|████████████████████████████████████████████████████████████████████████████████████████████████████████▋                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          | 5/46 [00:49<06:30,  9.52s/it]

{
    "resource_metrics": [
        {
            "resource": {
                "attributes": {
                    "telemetry.sdk.language": "python",
                    "telemetry.sdk.name": "openlit",
                    "telemetry.sdk.version": "1.39.1",
                    "service.name": "default",
                    "deployment.environment": "default"
                },
                "schema_url": ""
            },
            "scope_metrics": [
                {
                    "scope": {
                        "name": "opentelemetry.instrumentation.httpx",
                        "version": "0.60b1",
                        "schema_url": "https://opentelemetry.io/schemas/1.11.0",
                        "attributes": null
                    },
                    "metrics": [
                        {
                            "name": "http.client.duration",
                            "description": "measures the duration of the outbound HTTP request",
           

 26%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       | 12/46 [01:49<04:46,  8.42s/it]

{
    "resource_metrics": [
        {
            "resource": {
                "attributes": {
                    "telemetry.sdk.language": "python",
                    "telemetry.sdk.name": "openlit",
                    "telemetry.sdk.version": "1.39.1",
                    "service.name": "default",
                    "deployment.environment": "default"
                },
                "schema_url": ""
            },
            "scope_metrics": [
                {
                    "scope": {
                        "name": "opentelemetry.instrumentation.httpx",
                        "version": "0.60b1",
                        "schema_url": "https://opentelemetry.io/schemas/1.11.0",
                        "attributes": null
                    },
                    "metrics": [
                        {
                            "name": "http.client.duration",
                            "description": "measures the duration of the outbound HTTP request",
           

...

 83%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋                                                                                                                                                                       | 38/46 [05:49<01:14,  9.26s/it]

{
    "resource_metrics": [
        {
            "resource": {
                "attributes": {
                    "telemetry.sdk.language": "python",
                    "telemetry.sdk.name": "openlit",
                    "telemetry.sdk.version": "1.39.1",
                    "service.name": "default",
                    "deployment.environment": "default"
                },
                "schema_url": ""
            },
            "scope_metrics": [
                {
                    "scope": {
                        "name": "opentelemetry.instrumentation.httpx",
                        "version": "0.60b1",
                        "schema_url": "https://opentelemetry.io/schemas/1.11.0",
                        "attributes": null
                    },
                    "metrics": [
                        {
                            "name": "http.client.duration",
                            "description": "measures the duration of the outbound HTTP request",
           

 96%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                                         | 44/46 [06:47<00:19,  9.51s/it]

{
    "resource_metrics": [
        {
            "resource": {
                "attributes": {
                    "telemetry.sdk.language": "python",
                    "telemetry.sdk.name": "openlit",
                    "telemetry.sdk.version": "1.39.1",
                    "service.name": "default",
                    "deployment.environment": "default"
                },
                "schema_url": ""
            },
            "scope_metrics": [
                {
                    "scope": {
                        "name": "opentelemetry.instrumentation.httpx",
                        "version": "0.60b1",
                        "schema_url": "https://opentelemetry.io/schemas/1.11.0",
                        "attributes": null
                    },
                    "metrics": [
                        {
                            "name": "http.client.duration",
                            "description": "measures the duration of the outbound HTTP request",
           

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 46/46 [07:06<00:00,  9.27s/it]


In [10]:
with dspy.context(lm = lm_4o_mini.copy(cache = False, temperature = 0.4)):
    response = summarizer(financial_data = invoice_data, notes = compressed_notes)
    print(response)

Prediction(
    summary='The current Accounts Receivable status shows **three outstanding invoices** totaling **$36,240.50**, with the most overdue invoice (3830244) from **Oct 16, 2025**, now **36 days overdue**. The latest invoice (3849710) was submitted on **Nov 13, 2025**, and is due on **Dec 13, 2025**, while the most recent one (3869038) has just been delivered and is due on **Jan 09, 2026**. There have been no recent client interactions noted, indicating a potential gap in follow-up communication. It is advisable to reach out to the client to discuss the overdue invoice and confirm their receipt of the latest invoice to ensure timely payment.'
)


In [11]:
with dspy.context(lm = lm_41_mini.copy(cache = False, temperature = 0.4)):
    response = summarizer(financial_data = invoice_data, notes = compressed_notes)
    print(response)

Prediction(
    summary='The client currently has three outstanding invoices totaling **$34,240.50**, with the oldest invoice 3830244 overdue by **36 days** as of Nov 14, 2025, and invoice 3849710 recently overdue by **7 days** as of Dec 13, 2025. Invoice 3869038 was delivered on Dec 10, 2025, and is not yet due. There are no notes or recorded interactions to clarify reasons for delays or confirm payment plans. Immediate follow-up is recommended to address the significantly overdue invoice 3830244 and to confirm payment status on invoice 3849710 to prevent further aging.'
)


In [16]:
with dspy.context(lm = lm_5_mini):
    response = summarizer_cot(financial_data = invoice_data, notes = compressed_notes)
    print(response)

Prediction(
    reasoning='I parsed the provided CSV financial_data to extract three immutable AR line items: invoice numbers, invoice amounts, AR amounts, submitted_on and due_on dates, and days overdue. There were no entries in the notes section, so I did not assume any client interactions or internal updates beyond the financial facts. I computed the total outstanding by summing the AR amounts ($14,523.00 + $24,114.50 + $185.00 = $38,822.50) and identified the overdue status per row (84 and 20 days overdue as provided, and one invoice not yet overdue). Based on the instructions, I prioritized presenting a concise timeline-style summary with dates in the required format and offering clear next-step recommendations without inventing any interactions.',
    summary='**$38,822.50 outstanding** across three invoices: **#3801528 $14,523.00** (submitted Aug 28,2025; due Sep 27,2025 — **84 days overdue**), **#3841429 $24,114.50** (submitted Nov 10,2025; due Nov 30,2025 — **20 days overdue**